In [2]:
pip install gensim numpy pandas tensorflow scikit-learn

  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached wheel-0.48.0-py3-none-any.whl.metadata (2.3 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached namex-0.1.0-py3-none-any.whl

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [2]:
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics.pairwise import cosine_similarity

# COMMON FUNCTIONS
def clean(text):
    return re.findall(r'\b[a-z]+\b', str(text).lower())

def train(docs):
    return Word2Vec([clean(d) for d in docs],
                    vector_size=50, window=5,
                    min_count=1, sg=0, epochs=10)

def vector(text, model):
    words = [w for w in clean(text) if w in model.wv]
    return np.mean([model.wv[w] for w in words], axis=0) if words else np.zeros(50)


# 4A - IMDB CLASSIFICATION
(xtr, ytr), (xte, yte) = imdb.load_data(num_words=5000)
words = {v+3:k for k,v in imdb.get_word_index().items()}

def decode(x):
    return " ".join(words.get(i, "") for i in x if i > 3)

train_docs = [decode(x) for x in xtr]
test_docs = [decode(x) for x in xte]

m = train(train_docs)
Xtr = np.array([vector(d,m) for d in train_docs])
Xte = np.array([vector(d,m) for d in test_docs])

clf = Sequential([Dense(32, activation='relu', input_shape=(50,)),
                  Dense(1, activation='sigmoid')])
clf.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
clf.fit(Xtr, ytr, epochs=5, batch_size=32)
print("4A:", clf.evaluate(Xte, yte))


# 4B - TWITTER CBOW
tweets = pd.read_csv("twitter_dataset.csv")["Text"].dropna().astype(str).tolist()
m = train(tweets)
print("4B Vocabulary:", list(m.wv.index_to_key)[:10])
print("Similar words:", m.wv.most_similar(m.wv.index_to_key[0]))


# 4C - DOCUMENT 3 CBOW
doc = """The speed of transmission is an important difference between influenza and COVID-19. 
    Influenza has a shorter incubation period and serial interval than COVID-19. 
    The serial interval for COVID-19 is estimated to be 5-6 days, 
    while for influenza it is 3 days. This means influenza can spread faster.

    Transmission during the first 3-5 days of illness and pre-symptomatic 
    transmission are major drivers of influenza spread. COVID-19 can also 
    spread before symptoms appear, but this was considered a less significant 
    driver in the passage.

    The reproductive number for COVID-19 is estimated to be between 2 and 2.5, 
    higher than influenza. However, estimates depend on context and time"""
m = train([doc])
print("4C Vector:", vector(doc, m))
print("Vocabulary:", list(m.wv.index_to_key))


# 4D - RECOMMENDATION SYSTEM
m = train(tweets)
V = np.array([vector(t, m) for t in tweets])
q = vector(input("Enter search text: "), m).reshape(1, -1)
scores = cosine_similarity(q, V)[0]

for i in scores.argsort()[-5:][::-1]:
    print(tweets[i], scores[i])


Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.7899 - loss: 0.4638
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8233 - loss: 0.3969
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8253 - loss: 0.3911
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8284 - loss: 0.3888
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.8291 - loss: 0.3867
782/782 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8243 - loss: 0.3922
4A: [0.3922114968299866, 0.8243200182914734]
4B Vocabulary: ['hard', 'tax', 'out', 'add', 'when', 'my', 'high', 'maybe', 'food', 'yourself']
Similar words: [('yeah', 0.9572151303291321), ('field', 0.9529864192008972), ('action', 0.95100337266922), ('herself', 0.9507400989532471), ('report', 0.9506999254226685), ('stage', 0.9505793452262878), ('article', 0.9503740072250366), ('believe', 0.949972927570343), ('late', 0.9490431547164917), ('my', 0.9482819437980652)]
4C Vector: [-7.83928204e-04  3.47

Enter search text:  important


Major ahead laugh course. Report author share.
Easy off through traditional important because conference. 0.97061247
Husband traditional culture bit foreign. Offer actually process town success.
Would important specific after. Cover even get defense stand degree speech.
Our stand together action physical base store him. Billion admit large growth. 0.96943617
Practice specific discuss remember. Act seven fund list kitchen. List either meet TV southern relate travel common.
Brother they street yourself news Mrs. Economy risk piece across kid personal. Only important discover this blood single style. 0.96941245
Development prevent team question score base. Edge check indeed commercial impact.
Instead officer center serious. Theory table during energy ready matter white. 0.96904296
Read goal red travel. Join dream short forget operation. Traditional drug class rich create.
Increase health our easy. Happen special deal thousand fish during. Development matter laugh effect. 0.96896994
